# Amaliyot 5. Gradient tushish

5-interaktiv laboratoriyaga («Gradient tushish») hamroh. Kurs mavzusi: tasniflash modellarini optimallashtirish, yoʻqotish funksiyalari va yaqinlashuv

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

## Logistik regressiya «noldan» ikki belgida
Ikki sinfni qoldiramiz va matnni 2D ga siqamiz, shunda ikki vazn (w1, w2) boʻyicha yoʻqotishlar landshaftini chizish mumkin.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from scipy.special import expit

sub = df[df.label.isin(["sport", "siyosat"])]
X = TfidfVectorizer().fit_transform(sub.text)
Z = StandardScaler().fit_transform(TruncatedSVD(2, random_state=0).fit_transform(X))
y = np.where(sub.label == "sport", 1, -1)
lam = 0.01

def loss(w): return np.mean(np.logaddexp(0, -y * (Z @ w))) + lam / 2 * w @ w
def grad(w, idx=slice(None)):
    s = y[idx] * (Z[idx] @ w)
    return -(Z[idx] * (y[idx] * expit(-s))[:, None]).mean(0) + lam * w

def run(method, lr, steps=60, w0=(-4.0, 4.0), beta=0.9, bs=8, seed=0):
    rng = np.random.default_rng(seed)
    w, v = np.array(w0), np.zeros(2)
    path, losses = [w.copy()], [loss(w)]
    for _ in range(steps):
        if method == "sgd":
            g = grad(w, rng.integers(0, len(y), bs))
        else:
            g = grad(w)
        if method == "momentum":
            v = beta * v + g; w = w - lr * v
        else:
            w = w - lr * g
        path.append(w.copy()); losses.append(loss(w))
    return np.array(path), losses

## 1. Oʻrganish tezligi taʼsiri

In [ ]:
for lr in [0.05, 0.5, 2, 8, 40]:
    plt.plot(run("gd", lr)[1], label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("qadam"); plt.ylabel("yoʻqotishlar"); plt.legend(); plt.title("Gradient tushish: yaqinlashish va uzoqlashish"); plt.show()

## 2. Yoʻqotishlar landshafti va GD, SGD, Momentum traektoriyalari

In [ ]:
g = np.linspace(-6, 6, 120)
W1, W2 = np.meshgrid(g, g)
L = np.array([[loss(np.array([a, b])) for a in g] for b in g])
plt.figure(figsize=(6.5, 6))
plt.contourf(W1, W2, L, levels=25, cmap="viridis_r"); plt.colorbar(label="yoʻqotishlar")
for m, lr, c in [("gd", 2, "white"), ("sgd", 2, "red"), ("momentum", 0.5, "orange")]:
    p, _ = run(m, lr); plt.plot(p[:, 0], p[:, 1], "-o", ms=3, color=c, label=m)
plt.legend(); plt.xlabel("w1"); plt.ylabel("w2"); plt.title("Optimallashtirish traektoriyalari"); plt.show()

## 3. sklearn bilan solishtirish
Bizning maqsad: log-loss oʻrtacha qiymati plus `lam/2 * ||w||²`. sklearn da bu `C = 1 / (lam * n)`.

In [ ]:
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(fit_intercept=False, C=1 / (lam * len(y)), tol=1e-8).fit(Z, y)
p, _ = run("gd", 2, steps=500)
print("sklearn:", sk.coef_[0].round(3), " bizning GD:", p[-1].round(3))

## Topshiriqlar

1. Tanlang `lr`, bunda GD eng tez yaqinlashadi, lekin uzoqlashmaydi. Bu `lr` landshaft egriligi bilan qanday bogʻliq?
2. Mini-batch hajmini oʻzgartiring `bs` — SGD uchun (1, 8, 64). Traektoriya shovqini va yaqinlashish tezligi qanday oʻzgaradi?
3. Yoʻqotish funksiyasini hinge ga almashtiring: `np.maximum(0, 1 - y*(Z@w))`. Uning gradienti qanday?